<a href="https://colab.research.google.com/github/parviza9999/Credit-Card-Fraud-Detection-/blob/main/notebooks/04_xgboost_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install xgboost -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    roc_curve,
    precision_recall_curve
)

from xgboost import XGBClassifier

import joblib
import warnings
warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [ ]:
file_path = "/content/drive/MyDrive/Interview Node/Capstone_Project/Fraud Detection GCP Capstone/data/raw/creditcard.csv"

df = pd.read_csv(file_path)

df.head()

In [ ]:
print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nClass distribution:")
print(df["Class"].value_counts())

print("\nClass percentage:")
print(df["Class"].value_counts(normalize=True) * 100)

In [ ]:
X = df.drop(columns=["Class"])
y = df["Class"]

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

print("\nTraining class distribution:")
print(y_train.value_counts(normalize=True) * 100)

print("\nTesting class distribution:")
print(y_test.value_counts(normalize=True) * 100)

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaled training shape:", X_train_scaled.shape)
print("Scaled testing shape:", X_test_scaled.shape)

In [ ]:
non_fraud_count = y_train.value_counts()[0]
fraud_count = y_train.value_counts()[1]

scale_pos_weight = non_fraud_count / fraud_count

print("Non-fraud training count:", non_fraud_count)
print("Fraud training count:", fraud_count)
print("scale_pos_weight:", scale_pos_weight)

In [ ]:
xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    objective="binary:logistic",
    eval_metric="aucpr",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

xgb_model.fit(X_train_scaled, y_train)

print("XGBoost model training complete.")

In [ ]:
y_pred = xgb_model.predict(X_test_scaled)
y_proba = xgb_model.predict_proba(X_test_scaled)[:, 1]

print("Predictions generated.")

In [ ]:
cm = confusion_matrix(y_test, y_pred)

cm_df = pd.DataFrame(
    cm,
    index=["Actual Non-Fraud", "Actual Fraud"],
    columns=["Predicted Non-Fraud", "Predicted Fraud"]
)

cm_df

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_proba)
pr_auc = average_precision_score(y_test, y_proba)

xgb_results = pd.DataFrame({
    "Model": ["XGBoost"],
    "Accuracy": [accuracy],
    "Precision": [precision],
    "Recall": [recall],
    "F1 Score": [f1],
    "ROC-AUC": [roc_auc],
    "PR-AUC": [pr_auc]
})

xgb_results

In [ ]:
print(classification_report(y_test, y_pred, target_names=["Non-Fraud", "Fraud"]))

In [ ]:
fpr, tpr, roc_thresholds = roc_curve(y_test, y_proba)

plt.figure(figsize=(8, 5))
plt.plot(fpr, tpr, label=f"XGBoost ROC-AUC = {roc_auc:.4f}")
plt.plot([0, 1], [0, 1], linestyle="--", label="Random Classifier")
plt.title("XGBoost ROC Curve")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.show()

In [ ]:
precision_curve, recall_curve, pr_thresholds = precision_recall_curve(y_test, y_proba)

plt.figure(figsize=(8, 5))
plt.plot(recall_curve, precision_curve, label=f"XGBoost PR-AUC = {pr_auc:.4f}")
plt.title("XGBoost Precision-Recall Curve")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.legend()
plt.show()

In [ ]:
feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": xgb_model.feature_importances_
}).sort_values("Importance", ascending=False)

feature_importance.head(15)

In [ ]:
top_features = feature_importance.head(15).sort_values("Importance")

plt.figure(figsize=(8, 6))
plt.barh(top_features["Feature"], top_features["Importance"])
plt.title("Top 15 XGBoost Feature Importances")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.show()

In [ ]:
thresholds = [0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90]

threshold_results = []

for threshold in thresholds:
    y_pred_threshold = (y_proba >= threshold).astype(int)

    threshold_results.append({
        "Threshold": threshold,
        "Precision": precision_score(y_test, y_pred_threshold, zero_division=0),
        "Recall": recall_score(y_test, y_pred_threshold, zero_division=0),
        "F1 Score": f1_score(y_test, y_pred_threshold, zero_division=0),
        "False Positives": confusion_matrix(y_test, y_pred_threshold)[0, 1],
        "False Negatives": confusion_matrix(y_test, y_pred_threshold)[1, 0]
    })

threshold_results_df = pd.DataFrame(threshold_results)

threshold_results_df

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(threshold_results_df["Threshold"], threshold_results_df["Precision"], marker="o", label="Precision")
plt.plot(threshold_results_df["Threshold"], threshold_results_df["Recall"], marker="o", label="Recall")
plt.plot(threshold_results_df["Threshold"], threshold_results_df["F1 Score"], marker="o", label="F1 Score")
plt.title("XGBoost Threshold Tuning")
plt.xlabel("Classification Threshold")
plt.ylabel("Score")
plt.legend()
plt.show()

In [ ]:
best_threshold_row = threshold_results_df.sort_values("F1 Score", ascending=False).iloc[0]

best_threshold = best_threshold_row["Threshold"]

best_threshold_row

In [ ]:
y_pred_best = (y_proba >= best_threshold).astype(int)

final_cm = confusion_matrix(y_test, y_pred_best)

final_cm_df = pd.DataFrame(
    final_cm,
    index=["Actual Non-Fraud", "Actual Fraud"],
    columns=["Predicted Non-Fraud", "Predicted Fraud"]
)

final_cm_df

In [ ]:
final_accuracy = accuracy_score(y_test, y_pred_best)
final_precision = precision_score(y_test, y_pred_best)
final_recall = recall_score(y_test, y_pred_best)
final_f1 = f1_score(y_test, y_pred_best)
final_roc_auc = roc_auc_score(y_test, y_proba)
final_pr_auc = average_precision_score(y_test, y_proba)

xgb_tuned_results = pd.DataFrame({
    "Model": ["XGBoost Tuned Threshold"],
    "Threshold": [best_threshold],
    "Accuracy": [final_accuracy],
    "Precision": [final_precision],
    "Recall": [final_recall],
    "F1 Score": [final_f1],
    "ROC-AUC": [final_roc_auc],
    "PR-AUC": [final_pr_auc]
})

xgb_tuned_results

In [ ]:
model_output_folder = "/content/drive/MyDrive/Interview Node/Capstone_Project/Fraud Detection GCP Capstone/model_artifacts"
report_output_folder = "/content/drive/MyDrive/Interview Node/Capstone_Project/Fraud Detection GCP Capstone/reports"

import os
os.makedirs(model_output_folder, exist_ok=True)
os.makedirs(report_output_folder, exist_ok=True)

joblib.dump(xgb_model, f"{model_output_folder}/xgboost_fraud_model.joblib")
joblib.dump(scaler, f"{model_output_folder}/xgboost_scaler.joblib")

xgb_results.to_csv(f"{report_output_folder}/xgboost_default_results.csv", index=False)
xgb_tuned_results.to_csv(f"{report_output_folder}/xgboost_tuned_results.csv", index=False)
threshold_results_df.to_csv(f"{report_output_folder}/xgboost_threshold_results.csv", index=False)
feature_importance.to_csv(f"{report_output_folder}/xgboost_feature_importance.csv", index=False)

print("Model, scaler, and reports saved successfully.")

## XGBoost Model Summary

The XGBoost supervised classification model was trained using the Credit Card Fraud Detection dataset. Because the dataset is highly imbalanced, the model used `scale_pos_weight` to increase the penalty for misclassifying fraud transactions.

The evaluation focused on fraud-sensitive metrics including precision, recall, F1-score, ROC-AUC, PR-AUC, and the confusion matrix. Accuracy was reported but was not treated as the primary success metric because fraudulent transactions represent only a very small percentage of the dataset.

Threshold tuning was also performed because the default classification threshold of 0.50 may not provide the best balance between fraud recall and false positives. The recommended threshold was selected based on F1-score, but the final deployment threshold may be adjusted depending on the business priority between catching more fraud and reducing false alarms.

Key outputs saved:
- XGBoost trained model
- StandardScaler object
- Default model metrics
- Tuned-threshold metrics
- Threshold comparison table
- Feature importance table